# Project — River water temperature — *Hub'Eau, température des cours d'eau*
**MSA-DATI07-03 · Advanced EDA & Data Collection** · difficulty ⭐⭐ medium

This is your **project notebook**. It already follows the six-section structure from Session 6 and contains working collection code for this API. Your job: fill every `# TODO` and every *TODO* markdown cell, on **real data you collected yourselves**.

| # | Section | Answers the question… |
|---|---|---|
| 1 | Context & provenance | Where does this data come from, and what question is it for? |
| 2 | Setup & parameters | Can someone else run this and get the same thing? |
| 3 | Collection, ingestion & schema | Did we collect it properly, and is it what we think it is? |
| 4 | Data quality | What is missing, duplicated or weird — and what did we decide? |
| 5 | Exploration | What does the data say? Every figure captioned. |
| 6 | Findings & next steps | What can a modeller trust, and what should they test? |

**Deliverables:** this notebook (Restart & Run All passes, same fingerprint twice) · `requirements.txt` · the 2-page data-quality report (`DATA_QUALITY_REPORT_TEMPLATE.md`) · regular Git commits.

**Possible questions** (pick one or write your own — one per group):
- How does river temperature vary by season and between stations of a département?
- Are summer peaks getting higher across the years you collected?
- How continuous are the sensors: where and when are the gaps?

---
## 1 · Context & provenance

| Field | Your answer |
|---|---|
| **Business question** | How does river water temperature vary by season and between the stations of one département? We describe the seasonal cycle (monthly / seasonal levels and ranges) and compare stations within the same département, to tell a modeller which effects (season, station) any temperature model must account for. |
| **Source** | Hub'Eau (hubeau.eaufrance.fr), French water data API, endpoint `temperature/chronique` (continuous water temperature measured by in-river sensors) |
| **Dataset page / API docs** | https://hubeau.eaufrance.fr/page/api-temperature-continu |
| **Collected by / when** | Group: Abhy, Andrea, Kaled, Sofyan · collection run on *TO CONFIRM (date printed by `collected_at` in Section 3)* |
| **Collection method** | REST API, paginated (`next` link), cached with `requests-cache` (1 week), retry with backoff on 429/5xx, 0.5 s pause between network calls (see Section 3) |
| **Filters applied** | `code_departement = 35` · `date_debut_mesure = 2023-01-01` · `date_fin_mesure = 2023-12-31` · `size = 1000` per page · cap `MAX_ROWS = 20,000` *(TO CONFIRM: copy the final values from the parameters cell)* |
| **Licence / constraints** | Licence Ouverte / Etalab 2.0 (free reuse with attribution: "Source: Hub'Eau") — *TO CONFIRM on the dataset page* |
| **Unit of observation** | One row = one temperature measurement at one station, at one date and time *(TO CONFIRM on the data: check that (station, date, time) is unique)* |
| **Known limitations** | (1) **Paging cap**: Hub'Eau refuses to page deeper than ~20,000 rows, so a large query is silently truncated. (2) **Sensor gaps**: sensors fail or are removed, so some stations have missing days or months. (3) **Spikes**: a sensor out of the water (low flow, maintenance) records air temperature. (4) **Different sampling frequencies** between stations (e.g. hourly vs every few hours), so raw row counts are not comparable without resampling. (5) **Point measurements**: one sensor per station, so it says nothing about the rest of the river (depth, banks). (6) **One département, one year**: results do not generalise to other regions or years. |

---
## 2 · Setup & parameters
On Colab, run the install cell once per session.

In [ ]:
# Colab / fresh environment only
import importlib.util
missing = [p for p in ("pandera", "requests_cache") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q "pandera>=0.24" requests-cache

In [ ]:
# --- Parameters: the ONLY cell you should need to edit to rerun on new data ---
SEED = 42
DATA_DIR = "data"
REPORT_DIR = "reports"
CACHE_HOURS = 24 * 7                              # re-use API responses for a week
IQR_K = 1.5
MAX_MISSING_SHARE = 0.30

BASE_URL = "https://hubeau.eaufrance.fr/api/v1/temperature/chronique"
QUERY = {"code_departement": "35",               # change to your département
         "date_debut_mesure": "2023-01-01",
         "date_fin_mesure": "2023-12-31",
         "size": 1000}
MAX_ROWS = 20_000                                 # Hub'Eau refuses to page deeper than about 20,000 rows
PAUSE_S = 0.5

In [ ]:
import hashlib, json, random, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandera
import pandera.pandas as pa
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

random.seed(SEED); np.random.seed(SEED)
DATA, REPORTS = Path(DATA_DIR), Path(REPORT_DIR)
for p in (DATA / "raw", DATA / "processed", REPORTS):
    p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

for lib in (pd, np, sns, pandera, requests):
    print(f"{lib.__name__:<10}{lib.__version__}")

In [ ]:
# Write requirements.txt from the packages this notebook imports
from importlib.metadata import version
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "pandera", "pyarrow", "requests", "requests-cache"]
Path("requirements.txt").write_text("".join(f"{p}=={version(p)}\n" for p in pkgs))
print(Path("requirements.txt").read_text())

---
## 3 · Collection, ingestion & schema

### 3.1 A polite, cached HTTP session
- **Cache**: the same request is never sent twice (fast reruns, kind to the API).
- **Retry with backoff** on 429 / 5xx, respecting `Retry-After`.
- **Throttle**: pause between requests that actually hit the network.

In [ ]:
session = requests_cache.CachedSession("cache/api", expire_after=CACHE_HOURS * 3600)
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True)))
session.headers["User-Agent"] = "MSA-DATI07-student-project/1.0"

def get(url, params=None):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    if not getattr(r, "from_cache", False):
        time.sleep(PAUSE_S)
    return r

### 3.2 Collect

In [ ]:
def collect_next_link(url, params, items_key, max_rows=MAX_ROWS):
    """APIs that return the URL of the next page in the response ("next").
    We follow it until it disappears or we have enough rows."""
    rows, first = [], True
    while url and len(rows) < max_rows:
        r = get(url, params=params if first else None)   # the next link already contains the params
        page = r.json()
        if first:
            total = page.get("total", page.get("count"))
            print(f"API reports {total:,} matching rows" if total is not None else "total unknown")
        rows += page.get(items_key, [])
        url, first = page.get("next"), False
    return rows[:max_rows]

In [ ]:
collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
raw_rows = collect_next_link(BASE_URL, QUERY, items_key="data")
print(f"collected {len(raw_rows):,} rows at {collected_at}")

### 3.3 Flatten, save raw Parquet + README

In [ ]:
df_raw = pd.json_normalize(raw_rows, sep='_')

In [ ]:
raw_path = DATA / "raw" / "raw.parquet"
df_raw.to_parquet(raw_path, index=False)
(DATA / "raw" / "README.md").write_text(f"""# Raw data
- Source: see Section 1 of the notebook
- Collected at (UTC): {collected_at}
- Rows x columns: {df_raw.shape[0]} x {df_raw.shape[1]}
- Columns: {", ".join(df_raw.columns)}
""")
print(df_raw.shape)
df_raw.head()

In [ ]:
# First look: types, missing share, number of distinct values per column
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "share_missing": df_raw.isna().mean().round(3),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("share_missing", ascending=False)
overview

### 3.4 Schema-as-contract

*TODO (Exercise 3 of Session 6)*: write a `pa.DataFrameSchema` for the columns you keep. Requirements: every kept column with a dtype, `strict=True`, **at least 4 domain checks**, `nullable=True` only where justified in a comment. Validate the raw data with `lazy=True` and keep the failure summary for your report.

Columns to look at first (confirm the exact names in `df_raw.columns`): `code_station`, `libelle_station`, `date_mesure_temp`, `heure_mesure_temp`, `resultat`, `code_unite`, `libelle_cours_eau` (TO CONFIRM).

In [ ]:
# TODO: select the columns you keep, then write the contract
KEEP = list(df_raw.columns)          # TODO: replace by your selection (and log it as a decision)
df_sel = df_raw[KEEP].copy()

schema = None                        # TODO: replace None by pa.DataFrameSchema({...}, strict=True)

if schema is not None:
    try:
        schema.validate(df_sel, lazy=True)
        print("✅ raw data passes the contract")
    except pa.errors.SchemaErrors as err:
        fails = err.failure_cases
        print(f"❌ {len(fails)} violations")
        display(fails.groupby(["column", "check"]).size().rename("n").reset_index())

---
## 4 · Data quality — measure → decide → log

Every decision gets one `log(...)` call. The table at the end of this section goes into your report.

**What to look for in this dataset:**
- Gaps in each station's time series: sensor failure (MNAR?) or maintenance? Plot measurements per day per station.
- Spikes: a sensor out of the water reads air temperature. How do you detect it — jump size, comparison with neighbours?
- Different sampling frequencies between stations: resample before comparing.
- Units (`code_unite`): all °C? Check, don't assume.

In [ ]:
decisions = []
def log(column, issue, n_rows, decision, why):
    decisions.append(dict(column=column, issue=issue, n_rows=n_rows, decision=decision, why=why))

df = df_sel.copy()

### 4.1 Duplicates

In [ ]:
n_exact = int(df.astype(str).duplicated().sum())    # astype(str): some columns may hold dicts
print("exact duplicate rows:", n_exact)
# TODO: decide, apply, and log. Then look for near-duplicates (same entity, different id or spelling).

### 4.2 Impossible values

In [ ]:
# TODO: for each numeric column, what is physically or logically impossible? Count, decide, log.
df.describe().T

### 4.3 Missingness — mechanism before method

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0].rename("share_missing").to_frame()
miss["flag"] = np.where(miss.share_missing > MAX_MISSING_SHARE, "⚠️ high", "")
miss

In [ ]:
# TODO: for the 2–3 most important columns with missing values, test whether missingness
# depends on another observed column (MAR) — e.g.
# df.assign(m=df["<col>"].isna()).groupby("<other col>")["m"].mean()
# Then name the mechanism (MCAR / MAR / MNAR), decide, log.

### 4.4 Outliers — triage, don't delete

In [ ]:
# TODO: pick your key numeric column. Use an IQR fence (on a log scale if it is skewed),
# look at the flagged rows, separate errors from real extremes, decide per group, log.

In [ ]:
decision_table = pd.DataFrame(decisions, columns=["column", "issue", "n_rows", "decision", "why"])
decision_table.to_csv(REPORTS / "decision_table.csv", index=False)
decision_table

---
## 5 · Exploration — every figure gets a caption and a *so what*

In [ ]:
def show(fig, title, caption, so_what, name):
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.text(0.01, -0.04, f"{caption}\n→ {so_what}", ha="left", va="top", fontsize=9, wrap=True)
    fig.savefig(REPORTS / f"{name}.png", bbox_inches="tight", dpi=120)
    plt.show()

clean = df.copy()      # TODO: this should be your cleaned data after Section 4

### 5.1 Univariate

In [ ]:
# TODO: one figure showing the distribution of your key variable (ECDF before `.describe()`)
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_1")

### 5.2 Bivariate

In [ ]:
# TODO: one figure showing your key variable against 1–2 explanatory variables
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_2")

### 5.3 Multivariate

In [ ]:
# TODO: one figure showing the correlation structure, or a breakdown by two categories
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_3")

---
## 6 · Findings, what cannot be trusted, next steps

*TODO*

**Three findings** — each with the figure it comes from and what it means for modelling.
1.
2.
3.

**What cannot be trusted** — column, subset, and why.
-
-

**Five testable hypotheses** — variable, direction, population, test.
- H1:
- H2:
- H3:
- H4:
- H5:

### Final contract & fingerprint
*TODO*: write `clean_schema` for the data a modeller receives (it may differ from the raw schema). Then **Restart & Run All twice**: the fingerprint must be identical.

In [ ]:
clean_schema = None                  # TODO: pa.DataFrameSchema({...}, strict=True)
if clean_schema is not None:
    clean_schema.validate(clean, lazy=True)
    print("✅ clean data passes the contract")

clean.to_parquet(DATA / "processed" / "clean.parquet", index=False)
fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(clean.astype(str), index=False).values.tobytes()).hexdigest()[:12]
print("rows:", len(clean), "· fingerprint:", fingerprint)

> **Reproducibility note.** The fingerprint is stable between runs *thanks to the cache*: the API data changes over time, so a run without the cache (or after `CACHE_HOURS`) can give a different fingerprint. That is expected — record the collection date in Section 1 and in the README, and commit the raw Parquet if it is small enough.